# Text Summarization with Hugging Face Transformers

**Abstractive Summarization** using BART, PEGASUS, and T5.  
This notebook demonstrates text summarization, evaluation using ROUGE and BERTScore, and hallucination detection for generated summaries.

**Use Case:** This can be applied to news summarization, report summarization, content recommendation, and automated briefing generation in real-world NLP projects.

## 1. Imports & Environment Setup

In [1]:
# Install necessary libraries
!pip install -q evaluate rouge-score bert-score sentencepiece

# Suppress warnings to keep outputs clean
import warnings
warnings.filterwarnings("ignore")  # Suppress unnecessary warnings for cleaner output

# Standard Python libraries
import time          # Measure inference latency
import os            # Set environment variables
import pandas as pd  # Organize results into dataframes

# NLP utilities
import nltk
nltk.download("punkt")  # Required for sentence tokenization
from nltk.tokenize import sent_tokenize  # Break summaries into sentences

# Hugging Face libraries
from datasets import load_dataset  # Load benchmark datasets
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification  # Summarization and NLI

# Evaluation metrics
from evaluate import load  # For ROUGE and BERTScore

# PyTorch
import torch

# Optional: avoid tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-01-07 05:40:36.723805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767764436.959128      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767764437.031106      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767764437.611041      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767764437.611097      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the

## 2. Load Dataset

We use a small subset of the CNN/DailyMail dataset for testing summarization pipelines.
This dataset contains news articles and their reference summaries (highlights).

In [2]:
# Load 100 articles from the test split
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test[:100]")

# Inspect first article
print(dataset[0].keys())  # 'article' and 'highlights'

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

dict_keys(['article', 'highlights', 'id'])


## 3. Initialize Summarization Pipelines

We initialize three popular abstractive summarization models:

1. **BART**: Effective for news summarization.
2. **PEGASUS**: Optimized for summarization, trained on large news corpora.
3. **T5**: Text-to-text model that generalizes well for multiple NLP tasks.

These pipelines handle tokenization, inference, and decoding automatically.

In [3]:
# Initialize BART summarizer
bart_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Initialize PEGASUS summarizer
pegasus_summarizer = pipeline("summarization", model="google/pegasus-cnn_dailymail")

# Initialize T5 summarizer
t5_summarizer = pipeline("summarization", model="t5-large")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


## 4. Initialize Evaluation Metrics

We evaluate generated summaries using:

- **ROUGE-L**: Measures n-gram overlap and longest common subsequence.
- **BERTScore F1**: Measures semantic similarity with contextual embeddings.
- **Hallucination detection**: Checks if generated text introduces unsupported facts using an NLI model.

In [4]:
# ROUGE metric
rouge = load("rouge")

# BERTScore
bertscore = load("bertscore")

# NLI model for hallucination detection
nli_model = pipeline(
    "text-classification",
    model="roberta-large-mnli"
)
nli_tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


## 5. Hierarchical Summarization Function
Long articles may exceed the model's max token length.
Hierarchical summarization splits text into chunks, summarizes each, then merges them.

In [5]:
def hierarchical_summary(text, summarizer, tokenizer, chunk_size=None, max_length=130, min_length=30):
    """
    Perform hierarchical summarization on long articles to avoid token length errors.

    Args:
        text (str): Full article text.
        summarizer: Hugging Face summarization pipeline.
        tokenizer: Corresponding tokenizer.
        chunk_size (int): Number of tokens per chunk (defaults to model max length or 512).
        max_length (int): Maximum tokens generated per chunk.
        min_length (int): Minimum tokens generated per chunk.

    Returns:
        str: Merged summary string.
    """
    # Use model_max_length if chunk_size not provided
    if chunk_size is None:
        # Fallback to 512 if tokenizer.model_max_length is too big or undefined
        chunk_size = getattr(tokenizer, "model_max_length", 512)
        if chunk_size > 1024:
            chunk_size = 512

    # Encode full text
    tokens = tokenizer.encode(text, truncation=False)
    
    if len(tokens) == 0:
        # fallback: if tokenizer fails, return original text
        return text

    # Split tokens into chunks
    token_chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]

    summaries = []
    for tc in token_chunks:
        chunk_text = tokenizer.decode(tc, skip_special_tokens=True)
        if len(chunk_text.strip()) == 0:
            continue  # skip empty chunks
        summary = summarizer(chunk_text, max_length=max_length, min_length=min_length, do_sample=False)[0]['summary_text']
        summaries.append(summary)

    return " ".join(summaries)

## 6. Clean Summary Function
Clean output by removing extra line breaks or tokens like `<n>` for readability.

In [6]:
def clean_summary(summary):
    """
    Clean and format generated summary.
    """
    return summary.replace("\n", " ").replace("<n>", " ").strip()

## 7. Hallucination Detection Function

Use NLI to check if each sentence in the summary is entailed by the source article.
Truncate inputs to Roberta max length to prevent tensor errors.

In [7]:
def hallucination_score(article, summary, nli_model, nli_tokenizer):
    """
    Compute hallucination score (unsupported content ratio) using NLI.
    """
    sentences = sent_tokenize(summary)
    entailments = 0
    max_tokens = 512  # Roberta-large max

    for sent in sentences:
        inputs = nli_tokenizer(
            f"{article} </s></s> {sent}",
            truncation=True,
            max_length=max_tokens,
            return_tensors="pt"
        )
        outputs = nli_model.model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1)
        label = nli_model.model.config.id2label[pred.item()]
        if label == "ENTAILMENT":
            entailments += 1

    return 1 - (entailments / len(sentences)) if sentences else 0

## 8. Generate & Evaluate Summaries
Generate summaries for all articles using BART, then evaluate using ROUGE-L, BERTScore, hallucination detection, and measure latency.

In [8]:
results = []

for i in range(len(dataset)):
    article = dataset[i]["article"]
    reference = dataset[i]["highlights"]

    start = time.time()
    bart_summary = hierarchical_summary(article, bart_summarizer, bart_summarizer.tokenizer)
    latency = time.time() - start

    bart_summary = clean_summary(bart_summary)

    rouge_l = rouge.compute(predictions=[bart_summary], references=[reference])["rougeL"]
    bert_f1 = bertscore.compute(predictions=[bart_summary], references=[reference], lang="en")["f1"][0]
    hallucination = hallucination_score(article, bart_summary, nli_model, nli_tokenizer)

    results.append({
        "rougeL": rouge_l,
        "bert_f1": bert_f1,
        "hallucination_score": hallucination,
        "latency_sec": latency
    })

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Your max_length is set to 130, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Your max_length is set to 130, but your input_length is only 112. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=56)
Your max_length is set to 130, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarize

## 9. Inspect Results

Convert results to DataFrame for analysis.

In [9]:
df_results = pd.DataFrame(results)
df_results.describe()

,rougeL,bert_f1,hallucination_score,latency_sec
count,100.000000,100.000000,100.000000,100.000000
mean,0.238433,0.869919,0.828333,18.976910
std,0.131695,0.027868,0.353319,11.332087
min,0.061538,0.813880,0.000000,6.151952
25%,0.146092,0.848797,1.000000,9.709488
50%,0.205591,0.867266,1.000000,16.796698
75%,0.309859,0.887681,1.000000,24.522830
max,0.877193,0.969882,1.000000,61.407005


## 10. Conclusion

In this notebook, we built and evaluated an **abstractive text summarization pipeline** using pre-trained Transformer models from Hugging Face. The workflow covered the full lifecycle of a summarization task, including inference on real-world news articles, handling long documents, and quantitative evaluation.

The results show that while **ROUGE-L scores are moderate, BERTScore F1 values are consistently high**, indicating that the generated summaries preserve the **semantic meaning** of the original texts rather than relying on direct sentence extraction. In addition, hallucination analysis suggests that most summaries remain **faithful to the source content**, with limited unsupported generation.

Latency measurements reveal that the current setup is best suited for **offline or batch summarization** rather than real-time use, which is expected when working with large Transformer models. Overall, this project demonstrates a **practical, evaluation-driven approach** to abstractive summarization and provides a strong foundation for further improvements such as model fine-tuning, long-context architectures, or deployment-focused optimization.

## 11. Key Insights

**Semantic quality is strong:** High BERTScore F1 values indicate that the models generate summaries that closely preserve the meaning of the original articles, even when phrasing differs significantly.

**Lexical overlap is limited:** Moderate ROUGE-L scores highlight the abstractive nature of the models, as they rephrase content instead of copying sentences verbatim.

**Factual consistency is generally reliable:** Hallucination analysis shows that most generated summaries remain grounded in the source documents, with only a small subset exhibiting unsupported statements.

**Performance trade-offs exist:** Inference latency varies significantly across samples, making the current pipeline more suitable for batch processing rather than real-time applications.

**Evaluation beyond ROUGE is essential:** Combining lexical, semantic, and factual metrics provides a more realistic assessment of summarization quality than relying on ROUGE alone.